# 二、经营指标体系设计与SQL分析
建立一套有业务意义的电商经营指标体系。
## 2.1 建立经营指标体系
将项目的经营指标体系划分为 六大分析模块、约 30 个核心指标
| 模块 | 核心 KPI | 辅助分析 |
|---|---|---|
| 整体经营 | 销售额、有效订单量、客单价、取消率 | 月度增长、商品件数、支付金额 |
| 商品经营 | 商品销售额、毛利额、毛利率 | 品类、品牌、折扣、商品四象限 |
| 客户价值 | 客户数、复购率、客户平均消费金额 | RFM、客户分群、客户贡献 |
| 客户留存 | 次月留存率、各期留存率 | Cohort 热力图 |
| 支付与订单 | 支付方式占比、平均配送时长、平均评分 | 分期、支付金额、配送延迟 |
| 物流与卖家 | 卖家销售额、卖家订单量、区域分布 | 跨州订单、卖家表现 |
### 2.1.1 整体经营指标体系
判断平台整体经营规模、销售变化和订单表现。
| 指标 | 计算公式 | 数据表 | 说明 |
|---|---|---|---|
| 商品销售额 | Σ商品成交单价 × 商品数量（每行商品明细为一件时直接求和） | order_items | 衡量平台商品交易规模 |
| 订单总量 | COUNT(DISTINCT order_id) | order | 衡量订单规模 |
| 有效订单量 | COUNT(DISTINCT order_id)，筛选已发货订单 | order | 衡量有效履约订单规模 |
| 取消订单量 | COUNT(DISTINCT order_id)，筛选已取消订单 | order | 衡量订单取消规模 |
| 订单取消率 | 取消订单量 / 订单总量 × 100% | order | 衡量订单取消情况 |
| 客户总数 | COUNT(DISTINCT customer_unique_id) | customers | 衡量实际客户规模 |
| 客单价（AOV） | 商品销售额 / 有效订单量 | order、order_items | 衡量每笔有效订单平均商品消费金额 |
| 平均订单商品件数 | 商品总件数 / 有效订单量 | order_items、order | 衡量订单商品购买数量 |
| 月度销售增长率 | （本月销售额－上月销售额）/ 上月销售额 × 100% | order、order_items | 观察销售增长情况 |
| 平均订单支付金额 | 支付总金额 / 支付订单数 | order_payments | 衡量客户实际支付规模 |


In [ ]:
# 2.2.1整体经营指标体系
# 1. 商品销售额
select sum(oi.price) as total_price
from order_items oi
join orders o on oi.order_id = o.order_id
where order_status = 'delivered';
# 2. 订单总量
select count(distinct orders.order_id) as total_orders
from orders;
# 3. 有效订单量
select count(distinct orders.order_id) as vaild_orders
from orders
where order_status in ('shipped', 'delivered');
# 4. 取消订单量
select count(distinct orders.order_id) as vaild_orders
from orders
where order_status in ('canceled');
# 5. 订单取消率
select round(count(distinct case when orders.order_status = 'canceled' then order_id end)*100/
nullif(count(distinct orders.order_id),0),2) as vaild_per
from orders;
# 6. 客户总数
select count(distinct customers.customer_unique_id) as total_cus
from customers;
# 7. 客单价
with sales as(select sum(oi.price) as total_price
from order_items oi
join orders o on oi.order_id = o.order_id
where order_status = 'delivered'),

vaildorders as (select count(distinct orders.order_id) as vaild_orders
from orders
where order_status in ('shipped', 'delivered'))
select round(s.total_price/v.vaild_orders,2) as aov
from sales s
cross join vaildorders v;

# 8. 平均订单商品总数
with num_items as (select count(*) as total_items from order_items),
vaildorders as (select count(distinct orders.order_id) as vaild_orders
from orders
where order_status in ('shipped', 'delivered'))
select round(n.total_items/v.vaild_orders,0) as ave_item
from num_items n
cross join vaildorders v;

# 9. 月度销售增长率

with monthlysales as(select date_format(o.order_purchase_timestamp, '%Y-%m') as sales_month,
sum(oi.price) as monthly_price
from orders o 
join order_items oi on o.order_id = oi.order_id
where o.order_status not in ('canceled', 'unavailable')
group by date_format(o.order_purchase_timestamp, '%Y-%m')),
saleswithlag as (select sales_month,monthly_price,lag(monthly_price) over (order by sales_month) as prev_month_sales
from monthlysales)
SELECT
    sales_month,
    ROUND(monthly_price, 2) AS monthly_price,
    ROUND(
        (monthly_price - prev_month_sales)
        / NULLIF(prev_month_sales, 0) * 100,
        2
    ) AS growth_rate_percentage
FROM saleswithlag
ORDER BY sales_month;

# 10.平均订单支付额

SELECT 
    ROUND(SUM(payment_value) / COUNT(DISTINCT order_id), 2) AS avg_order_payment
FROM order_payments;



### 2.1.2 客户价值分析指标
| 指标名称 | 计算公式 | 数据来源 | 分析目的 |
|---|---|---|---|
| 实际客户数 | COUNT(DISTINCT customer_unique_id) | customers、order | 统计实际客户数量 |
| 新客户数 | 首次购买时间落在统计周期内的客户数 | customers、order | 衡量新客户获取 |
| 复购客户数 | 购买订单数 ≥ 2 的客户数 | customers、order | 衡量客户重复购买情况 |
| 客户复购率 | 复购客户数 / 有购买行为客户数 × 100% | customers、order | 衡量客户复购情况 |
| 客户平均消费金额 | 商品销售额 / 实际购买客户数 | customers、order、order_items | 衡量客户平均消费水平 |
| 客户平均购买频次 | 有效订单量 / 实际购买客户数 | customers、order | 衡量客户购买频率 |
| 客户生命周期价值（简化） | 客户历史累计商品消费金额 | customers、order、order_items | 衡量客户累计贡献 |
| R（Recency） | 统计截止日期－客户最近一次购买日期 | customers、order | 衡量客户最近购买活跃程度 |
| F（Frequency） | 客户在统计周期内的有效订单数量 | customers、order | 衡量客户购买频次 |
| M（Monetary） | 客户在统计周期内的累计商品消费金额 | customers、order、order_items | 衡量客户消费贡献 |


**RFM 维度设计**

| 维度 | 含义 | 评分方向 |
|---|---|---|
| R | 最近一次购买距今的时间 | 越短分数越高 |
| F | 购买频次 | 越高分数越高 |
| M | 累计消费金额 | 越高分数越高 |

**RFM 客户分群**

| 客户群体 | 典型特征 | 运营建议 |
|---|---|---|
| 重要价值客户 | R、F、M 均较高 | 重点维护、会员权益 |
| 重要发展客户 | R 较高，F、M 有提升空间 | 促进复购和消费升级 |
| 重要保持客户 | F、M 较高，但 R 较低 | 定向召回 |
| 一般价值客户 | 消费频次和金额较低 | 低成本促活 |
| 流失风险客户 | 长时间未购买 | 召回优惠、定向营销 |


In [ ]:
# 2.1.2 商品经营分析指标
# 定义临时变量
SET @start_date = '2019-01-01';
SET @end_date   = '2026-01-01';
SET @as_of_date = '2026-01-01';
## 有效订单
with vaild_orders as (select o.order_id,c.customer_unique_id,date(o.order_purchase_timestamp) as purchase_date
from orders o
join customers c on c.customer_id = o.customer_id
where o.order_status not in ('canceled', 'unavailable')),
order_amt as (select order_items.order_id, sum(price) as item_amt
from order_items
group by order_id),
vaild_order_amt as (select v.order_id,v.customer_unique_id,v.purchase_date, coalesce(a.item_amt,0) as item_amount
from vaild_orders v
left join order_amt a on a.order_id = v.order_id),
period_order as (select * from vaild_order_amt
where purchase_date >= @start_date and purchase_date < @end_date),
period_customer as (select customer_unique_id,count(distinct order_id) as freq,sum(item_amount) as monetary
from period_order group by customer_unique_id),
first_purchase as(select customer_unique_id,min(purchase_date) as first_purchase_date
from vaild_order_amt group by customer_unique_id),
new_customer as (select customer_unique_id from first_purchase
where first_purchase_date >= @start_date
and first_purchase_date <= @end_date),
repurchase_customer as (select customer_unique_id from period_customer 
where freq >=2),
clv as (select customer_unique_id, sum(item_amount) as clv_amount, max(purchase_date) as lastest_pur_date
from vaild_order_amt
group by customer_unique_id)
# 1. 有效客户数
select count(*) as real_customers from period_customer
# 2. 新客户数
union all
select count(*) as new_customers from new_customer
# 3. 复购客户数
union all
select count(*) as repurchase_customers from repurchase_customer
# 4. 客户复购率
union all
select round((select count(*) as repurchase_customers from repurchase_customer)*100/
(select count(*) as real_customers from period_customer),2) as rep_per
# 5. 客户平均消费金额
union all
select round((select sum(monetary) from period_customer)/
(select count(*) as real_customers from period_customer),2) as ave_price
# 6. 客户平均购买频次
union all
select round((select sum(freq) from period_customer)/ 
nullif((select count(*) from period_customer), 0), 2);


## 有效订单
with vaild_orders as (select o.order_id,c.customer_unique_id,date(o.order_purchase_timestamp) as purchase_date
from orders o
join customers c on c.customer_id = o.customer_id
where o.order_status not in ('canceled', 'unavailable')),
order_amt as (select order_items.order_id, sum(price) as item_amt
from order_items
group by order_id),
vaild_order_amt as (select v.order_id,v.customer_unique_id,v.purchase_date, coalesce(a.item_amt,0) as item_amount
from vaild_orders v
left join order_amt a on a.order_id = v.order_id),
period_order as (select * from vaild_order_amt
where purchase_date >= @start_date and purchase_date < @end_date),
period_customer as (select customer_unique_id,count(distinct order_id) as freq,sum(item_amount) as monetary
from period_order group by customer_unique_id),
first_purchase as(select customer_unique_id,min(purchase_date) as first_purchase_date
from vaild_order_amt group by customer_unique_id),
new_customer as (select customer_unique_id from first_purchase
where first_purchase_date >= @start_date
and first_purchase_date <= @end_date),
repurchase_customer as (select customer_unique_id from period_customer 
where freq >=2),
clv as (select customer_unique_id, sum(item_amount) as clv_amount, max(purchase_date) as lastest_pur_date
from vaild_order_amt
group by customer_unique_id)
# 7. 客户生命周期价值
SELECT
    p.customer_unique_id,
    DATEDIFF(@as_of_date, h.lastest_pur_date) AS R,
    p.freq AS F,
    p.monetary AS M,
    h.clv_amount AS CLV
FROM period_customer p
JOIN clv h
    ON p.customer_unique_id = h.customer_unique_id
ORDER BY R ASC, F DESC, M DESC;

### 2.1.3 客户留存分析指标

| 指标名称 | 计算公式 | 数据来源 | 分析目的 |
|---|---|---|---|
| 首购客户数 | 某月首次购买客户数量 | customers、order | 衡量新客户获取 |
| 次月留存客户数 | 某月首购客户中，下月再次购买的客户数 | customers、order | 衡量次月留存 |
| 次月留存率 | 次月留存客户数 / 首购客户数 × 100% | customers、order | 衡量客户次月留存表现 |
| 第N月留存率 | 首购月份为第0期的客户中，第N期仍有购买行为的客户数 / 首购客户数 | customers、order | 衡量长期留存 |
| 客户流失率 | 观察期内未复购客户数 / 符合观察条件的客户数 | customers、order | 衡量客户流失情况 |


Cohort 分析设计
按照客户第一次购买的月份划分 Cohort。

In [ ]:
with vaild_order as (select o.order_id,c.customer_unique_id,date(o.order_purchase_timestamp) as purchase_date
from orders o
join customers c on c.customer_id = o.customer_id
where o.order_status not in ('canceled', 'unavailable')),

first_month as (select customer_unique_id, min(purchase_date) as first_purchase_date, 
date_format(min(purchase_date),'%Y-%m-01') as cohort_month
from vaild_order
group by customer_unique_id),

customer_order_month as(select v.customer_unique_id,f.cohort_month,date_format(v.purchase_date,'%Y-%m-01') as order_month,
timestampdiff(month,str_to_date(f.cohort_month,'%Y-%m-%d'),str_to_date(date_format(v.purchase_date,'%Y-%m-01'),'%Y-%m-%d')) as month_index
from vaild_order v
join first_month f on v.customer_unique_id = f.customer_unique_id),

cohort_size as(select cohort_month, count(distinct customer_unique_id) as first_buy_customers
from first_month
WITH customer_month AS (
    -- 1. 提取每个客户实际购买过的月份，同一客户同一月份只保留一条
    SELECT DISTINCT
        c.customer_unique_id,
        DATE_SUB(
            DATE(o.order_purchase_timestamp),
            INTERVAL (DAYOFMONTH(o.order_purchase_timestamp) - 1) DAY
        ) AS order_month
    FROM orders o
    JOIN customers c
        ON c.customer_id = o.customer_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
),

first_month AS (
    -- 2. 计算每个客户的首次购买月份
    SELECT
        customer_unique_id,
        MIN(order_month) AS cohort_month
    FROM customer_month
    GROUP BY customer_unique_id
),

cohort_size AS (
    -- 3. 计算每个月首次购买的客户数
    SELECT
        cohort_month,
        COUNT(*) AS first_buy_customers
    FROM first_month
    GROUP BY cohort_month
),

cohort_activity AS (
    -- 4. 计算客户购买月份距离首次购买月份的间隔
    SELECT
        cm.customer_unique_id,
        fm.cohort_month,
        TIMESTAMPDIFF(
            MONTH,
            fm.cohort_month,
            cm.order_month
        ) AS month_index
    FROM customer_month cm
    JOIN first_month fm
        ON cm.customer_unique_id = fm.customer_unique_id
),

retention AS (
    -- 5. 统计每个 Cohort 在各月份的留存客户数
    SELECT
        cohort_month,
        month_index,
        COUNT(*) AS retained_customers
    FROM cohort_activity
    GROUP BY cohort_month, month_index
)

-- 6. 计算留存率
SELECT
    r.cohort_month,
    r.month_index,
    r.retained_customers,
    c.first_buy_customers,
    ROUND(
        r.retained_customers * 100.0 /
        NULLIF(c.first_buy_customers, 0),
        2
    ) AS retention_rate
FROM retention r
JOIN cohort_size c
    ON r.cohort_month = c.cohort_month
ORDER BY r.cohort_month, r.month_index;

group by cohort_month),

retention as (select cohort_month,month_index,count(distinct customer_unique_id) as retained_customers
from customer_order_month
group by cohort_month, month_index)

select
    r.cohort_month,
    r.month_index,
    r.retained_customers,
    c.first_buy_customers,
    round(r.retained_customers* 100 / nullif(c.first_buy_customers,0) , 2) AS retention_rate
from retention r
join cohort_size c
  on r.cohort_month = c.cohort_month
order by r.cohort_month, r.month_index;


### 2.1.4 支付与订单分析指标

| 指标名称 | 计算公式 | 数据来源 | 分析目的 |
|---|---|---|---|
| 各支付方式订单数 | 按 payment_type 统计 DISTINCT order_id | order_payments | 了解支付方式偏好 |
| 各支付方式使用占比 | 使用某支付方式的订单数 / 有支付记录的订单数 | order_payments | 比较支付方式结构 |
| 各支付方式支付金额 | 按 payment_type 汇总 payment_value | order_payments | 比较不同支付方式交易规模 |
| 平均分期期数 | AVG(payment_installments) | order_payments | 分析消费者分期习惯 |
| 多次支付订单数 | COUNT(DISTINCT order_id)，筛选 payment_sequential 大于1的订单 | order_payments | 识别多笔支付订单 |
| 支付审批时长 | order_approved_at－order_purchase_timestamp | order | 分析付款审批效率 |
| 发货等待时间 | order_delivered_carrier_date－order_approved_at | order | 衡量订单发货效率 |
| 物流配送时长 | order_delivered_customer_date－order_delivered_carrier_date | order | 衡量物流配送效率 |
| 配送延迟天数 | 实际送达时间－预计送达时间 | order | 衡量履约表现 |
| 平均客户评分 | AVG(review_score) | order_reviews | 衡量客户满意度 |

In [ ]:
with
-- 1. 各支付方式订单数
pay_orders as (select payment_type, count(distinct order_id) as order_cnt
from order_payments
group by payment_type),
-- 2. 有支付记录的总订单数
total_orders as (select count(distinct order_id) as total_order_cnt
from order_payments),
-- 3. 各支付方式支付金额，总数与平均
pay_amount as (select payment_type,count(*) as payment_record_cnt,
sum(payment_value) as total_payment_value,avg(payment_value) as avg_payment_value
from order_payments
group by payment_type),
-- 4. 平均分期期数
order_inst as (select order_id,max(payment_type) as payment_type,max(payment_installments) as installments
from order_payments
where payment_installments > 0
group by order_id),
inst_by_type as (select payment_type,count(*) as inst_order_cnt,avg(installments) as avg_installments_by_order
from order_inst
group by payment_type),
-- 5. 多次支付订单
multi_pay_orders as (select distinct order_id
from order_payments
where payment_sequential > 1),
multi_by_type as (select p.payment_type,count(distinct p.order_id) as multi_payment_order_cnt
from order_payments p
join multi_pay_orders m on p.order_id = m.order_id
group by p.payment_type),
-- 6. 支付审批时长（小时）
approval_time as (select order_id,timestampdiff(second, order_purchase_timestamp, order_approved_at) / 3600.0 as approval_hours
from orders
where order_approved_at is not null
and order_purchase_timestamp is not null),
approval_summary as (select count(*) as approval_order_cnt,avg(approval_hours) as avg_approval_hours
from approval_time),
-- 7. 发货等待时间（小时）
ship_wait as (select order_id,timestampdiff(second, order_approved_at, order_delivered_carrier_date) / 3600.0 as ship_wait_hours
from orders
where order_approved_at is not null
and order_delivered_carrier_date is not null),
ship_summary as (select count(*) as ship_order_cnt,avg(ship_wait_hours) as avg_ship_wait_hours
from ship_wait),
-- 8. 物流配送时长（天）
delivery_time as (select order_id,
timestampdiff(second, order_delivered_carrier_date, order_delivered_customer_date) / 86400.0 as delivery_days
from orders),
delivery_summary as (select count(*) as delivery_order_cnt,avg(delivery_days) as avg_delivery_days
from delivery_time),
-- 9. 配送延迟天数（正=延迟，负=提前）
delay_time as (select order_id,
timestampdiff(second, order_estimated_delivery_date, order_delivered_customer_date) / 86400.0 as delay_days,
	case
		when order_delivered_customer_date > order_estimated_delivery_date then 1
		else 0
	end as is_delayed
from orders),
delay_summary as (select count(*) as delivered_order_cnt,avg(delay_days) as avg_delay_days,
sum(is_delayed) * 100.0 / nullif(count(*), 0) as delayed_order_pct
from delay_time),
-- 10. 平均客户评分（按订单去重）
order_review as (select order_id,avg(review_score) as avg_score
from order_reviews
where review_score is not null
group by order_id),
review_summary as (select count(*) as review_order_cnt, avg(avg_score) as avg_review_score
from order_review)
-- 最终宽表：每行一个 payment_type
select
    p.payment_type,
    p.order_cnt,
    round(p.order_cnt * 100.0 / nullif(t.total_order_cnt, 0), 2) as usage_pct,
    round(a.total_payment_value, 2) as total_payment_value,
    round(a.avg_payment_value, 2) as avg_payment_value,
    round(i.avg_installments_by_order, 2) as avg_installments_by_order,
    coalesce(m.multi_payment_order_cnt, 0) as multi_payment_order_cnt,
    round(ap.avg_approval_hours, 2) as avg_approval_hours,
    round(sh.avg_ship_wait_hours, 2) as avg_ship_wait_hours,
    round(dl.avg_delivery_days, 2) as avg_delivery_days,
    round(de.avg_delay_days, 2) as avg_delay_days,
    round(de.delayed_order_pct, 2) as delayed_order_pct,
    round(rv.avg_review_score, 2) as avg_review_score
from pay_orders p
cross join total_orders t
cross join approval_summary ap
cross join ship_summary sh
cross join delivery_summary dl
cross join delay_summary de
cross join review_summary rv
left join pay_amount a on p.payment_type = a.payment_type
left join inst_by_type i on p.payment_type = i.payment_type
left join multi_by_type m on p.payment_type = m.payment_type
order by p.order_cnt desc;



### 2.1.5 物流与卖家分析指标

| 指标名称 | 计算公式 | 数据来源 | 分析目的 |
|---|---|---|---|
| 卖家总数 | COUNT(DISTINCT seller_id) | sellers | 衡量卖家规模 |
| 卖家订单量 | COUNT(DISTINCT order_id) | sellers、order_items | 衡量卖家订单贡献 |
| 卖家商品销售额 | SUM(order_items.price) | sellers、order_items | 衡量卖家销售规模 |
| 卖家商品毛利贡献 | Σ（商品成交价－商品成本） | sellers、order_items、products | 比较卖家关联商品的毛利贡献 |
| 卖家平均订单金额 | 卖家商品销售额 / 卖家有效订单数 | sellers、order_items、order | 比较卖家订单价值 |
| 卖家区域分布 | 按 seller_state 统计卖家数 | sellers | 分析卖家地域结构 |
| 客户区域分布 | 按 customer_state 统计客户数 | customers | 分析客户地域结构 |
| 跨州订单占比 | 客户州与卖家州不同的订单数 / 有效订单数 | customers、sellers、order_items、order | 分析跨区域交易情况 |
| 平均配送延迟 | 实际送达时间－预计送达时间 | order | 衡量平台整体配送表现 |

In [ ]:
with
-- 1. 有效订单
valid_orders as (select order_id
from orders
where order_status not in ('canceled', 'unavailable')),
-- 2. 卖家-订单销售额（每个卖家每个订单的商品金额）
seller_order_sales as (select oi.seller_id,oi.order_id,sum(oi.price) as order_sales
from order_items oi
join valid_orders vo on oi.order_id = vo.order_id
group by oi.seller_id, oi.order_id),
-- 3. 卖家汇总：订单量、销售额
seller_summary as (select seller_id,count(distinct order_id) as order_cnt,sum(order_sales) as total_sales
from seller_order_sales
group by seller_id),
-- 4. 卖家平均订单金额
seller_avg as (select seller_id,total_sales / nullif(order_cnt, 0) as avg_order_value
from seller_summary),
-- 5. 卖家总数
total_sellers as (select count(distinct seller_id) as total_seller_cnt
from sellers),
-- 6. 卖家区域分布
seller_by_state as (select seller_state,count(distinct seller_id) as seller_cnt
from sellers
group by seller_state),
-- 7. 客户区域分布（按 customer_unique_id 去重）
customer_by_state as (select customer_state,count(distinct customer_unique_id) as customer_cnt
from customers
group by customer_state),
-- 8. 订单级别：是否跨州（订单内任一卖家州与客户州不同即记为跨州）
order_cross_state as (
select oi.order_id,max(case when s.seller_state <> c.customer_state then 1 else 0 end) as is_cross_state
from order_items oi
join valid_orders vo on oi.order_id = vo.order_id
join orders o on oi.order_id = o.order_id
join customers c on o.customer_id = c.customer_id
join sellers s on oi.seller_id = s.seller_id
group by oi.order_id),
-- 9. 跨州订单占比
cross_state_pct as (select sum(is_cross_state) * 100.0 / nullif(count(*), 0) as cross_state_order_pct
from order_cross_state),
delivery_delay AS (
    SELECT
        AVG(
            TIMESTAMPDIFF(
                SECOND,
                order_estimated_delivery_date,
                order_delivered_customer_date
            ) / 86400.0
        ) AS avg_delay_days
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
)
-- 最终宽表：每行一个 seller
select
    s.seller_id,
    s.seller_state,
    ss.order_cnt,
    round(ss.total_sales, 2) as total_sales,
    ROUND(dd.avg_delay_days, 2) AS avg_delay_days,
    round(sa.avg_order_value, 2) as avg_order_value,
    sbs.seller_cnt as seller_cnt_in_state,
    cbs.customer_cnt as customer_cnt_in_state,
    round(csp.cross_state_order_pct, 2) as cross_state_order_pct,
    ts.total_seller_cnt
from sellers s
left join seller_summary ss on s.seller_id = ss.seller_id
left join seller_avg sa on s.seller_id = sa.seller_id
left join seller_by_state sbs on s.seller_state = sbs.seller_state
left join customer_by_state cbs on s.seller_state = cbs.customer_state
cross join cross_state_pct csp
cross join total_sellers ts
CROSS JOIN delivery_delay dd
order by ss.total_sales desc;